# 第 26 课｜训练检查点与断点续训

**本课问题：程序退出后，能否从原来那一步接着训练，而不是仅从同一组权重重新开始？**

交付物：本 Notebook 与同名 `.py` 脚本各自独立可用；仅依赖 PyTorch 与 Python 标准库。
不下载模型、语料，不读取上一课文件。默认在 CPU、FP32 上学习；GPU 是可选实测路径。

本课承接第 25 课的梯度累积和激活检查点，保留 RoPE、GQA、RMSNorm、SwiGLU，
**开启 Dropout**，并加入学习率调度、完整随机状态与显式数据游标。

**先掌握三层“恢复”：**

|层级|恢复后能做什么|本课定位|
|---|---|---|
|加载权重|继续推理或作为新训练起点|不足以精确续训|
|恢复优化状态|接上动量、学习率和 AMP scaler|必要但还不充分|
|接上完整执行状态|接上随机序列、下一批数据与计数|在同一受控环境中实测逐位一致|

**边界：单进程、单设备、固定数据、只在完整累积窗口结束保存。**
不支持窗口中途梯度恢复、多 worker 预取、分布式状态切片或跨环境逐位保证。

## 学习顺序

先读 1–4 节的小实验，再读 5–8 节的保存与恢复流程，最后运行连续/中断对照和独立练习。
大段“支持代码”只是使文件独立运行，不要求一次背下全部模型实现；可折叠阅读。

学习后应能解释：

- 为什么保存 `model.state_dict()` 还不够；
- 为什么再次设置 seed 不等于恢复中途 RNG 状态；
- 为什么恢复随机状态应放在对象重建之后；
- 为什么只记录 epoch/batch 编号可能无法恢复数据；
- 为什么保存成功与训练恢复正确是两件事。

**使用方式：** Python 3 内核，按顺序运行。已有 PyTorch 环境无需重装。
第一次接触某个代码单元，先预测输出，再执行。练习开关默认关闭，全部运行成功不代表完成了练习。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第26课：训练检查点与断点续训。独立运行，仅依赖 PyTorch 和标准库。

默认运行：同一受控环境中的连续训练 vs. 中断恢复实验，并运行故障对照。
CPU FP32 为主验证；可选 CUDA/AMP（需本机实测）。采用显式单进程数据游标。
只允许在完整梯度累积窗口结束、scaler/scheduler 已更新且 grad=None 时保存。
不支持窗口中途、DataLoader 多 worker 预取、DDP/FSDP 或跨环境逐位恢复。
导入本模块不会训练、写文件或修改全局线程/随机状态。
"""
from __future__ import annotations

# %% 01. 配置与参考资料
import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import sys
import tempfile
from contextlib import contextmanager
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Any

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel
from torch.utils.checkpoint import checkpoint

FORMAT_VERSION = 1
IMPLEMENTATION_ID = "lesson26-resume-v1"
PAD, BOS, EOS, IGNORE = 0, 1, 2, -100
SOURCES = {
    "general_checkpoint": "https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html",
    "optimizer_load_order": "https://docs.pytorch.org/docs/2.10/generated/torch.optim.Optimizer.load_state_dict.html",
    "reproducibility": "https://docs.pytorch.org/docs/2.10/notes/randomness.html",
    "rng": "https://docs.pytorch.org/docs/stable/generated/torch.get_rng_state.html",
    "amp_resume": "https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html",
    "amp_accumulation": "https://docs.pytorch.org/docs/2.10/notes/amp_examples.html",
    "serialization": "https://docs.pytorch.org/docs/stable/notes/serialization.html",
    "data_workers": "https://docs.pytorch.org/docs/2.10/data.html",
    "atomic_replace": "https://docs.python.org/3/library/os.html#os.replace",
}


def positive_int(value, name):
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数（不接受 bool）")


def choose_device(name):
    if name not in ("cpu", "cuda", "auto"):
        raise ValueError("device 必须是 cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用，不会静默回退；先使用 --device cpu")
    return torch.device(name)


@dataclass(frozen=True)
class TrainConfig:
    micro_size: int = 3
    accumulation_steps: int = 4
    lr: float = 0.003
    weight_decay: float = 0.01
    max_norm: float = 1.0
    scheduler_step: int = 3
    scheduler_gamma: float = 0.8
    seed: int = 2601
    precision: str = "fp32"
    activation_checkpointing: bool = True

    def __post_init__(self):
        for key in ("micro_size", "accumulation_steps", "scheduler_step", "seed"):
            positive_int(getattr(self, key), key)
        for key in ("lr", "max_norm", "scheduler_gamma"):
            if not math.isfinite(getattr(self, key)) or getattr(self, key) <= 0:
                raise ValueError(f"{key} 必须为正且有限")
        if self.scheduler_gamma > 1 or self.weight_decay < 0 or not math.isfinite(self.weight_decay):
            raise ValueError("本课要求 gamma<=1，weight_decay 非负且有限")
        if self.precision not in ("fp32", "amp_bf16", "amp_fp16"):
            raise ValueError("precision 非法")
        if type(self.activation_checkpointing) is not bool:
            raise ValueError("activation_checkpointing 必须为 bool")

In [2]:
# 修改这里即可选择本机实验；默认无需显卡。
DEVICE_NAME = "cpu"
PRECISION = "fp32"             # 可选 amp_bf16；amp_fp16 仅在本课 CUDA 训练中启用
TOTAL_WINDOWS = 16
INTERRUPT_AFTER = 7
OUTPUT_DIR = Path("lesson_26_outputs")

if DEVICE_NAME in ("cuda", "auto"):
    # CUDA 已经初始化过的内核，应重启后先设置此项。
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
DEVICE = choose_device(DEVICE_NAME)
print("PyTorch：", torch.__version__)
print("实际设备：", DEVICE)
print("CUDA 可用：", torch.cuda.is_available())
print("输出目录：", OUTPUT_DIR.resolve())

PyTorch： 2.10.0+cpu
实际设备： cpu
CUDA 可用： False
输出目录： /mnt/data/_lesson26_work/notebook_only/lesson_26_outputs


## 1. 检查点是训练状态，不只是权重

AdamW 除了参数，还维护梯度的一阶、二阶统计与步数。学习率调度器、GradScaler 也维护跨步骤的状态。
仅把权重载入一个全新的优化器，已经改变了下一次更新的规则。[S1][S2][S3]

本课保存下列字段，全部使用基础容器和 Tensor，不把自定义模型对象直接 pickle：

|字段|意义|
|---|---|
|`model`、`model_config`|参数、buffer 与模型结构配置|
|`optimizer`、`parameter_names`|动量、步数、参数组，以及注册顺序核对|
|`scheduler`|衰减阶段与配置|
|`scaler`|缩放因子、增长计数等；FP32/BF16 下为空状态|
|`rng`|Python、CPU PyTorch、所用 CUDA 环境中的随机状态|
|`data_cursor`|当前排列、下一条样本游标、epoch、独立 shuffle generator 状态|
|`progress`、`history`|尝试窗口数、真实更新数、样本/token 数以及逐窗口记录|
|配置、数据签名、环境、格式版本|发现误接数据、环境变化或实现版本不兼容|

`epoch` 不够：同一轮中还需要知道已经处理到哪个样本。
`seed` 不够：它指定随机序列起点，不是“当前读到第几个随机数”。

In [3]:
# %% 04. 随机状态、实验环境和独立快照

def seed_all(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def capture_rng(device):
    return {"python": random.getstate(), "torch_cpu": torch.get_rng_state().clone(),
            "torch_cuda": ([s.clone() for s in torch.cuda.get_rng_state_all()]
                           if device.type == "cuda" else [])}


def restore_rng(state, device):
    if set(state) != {"python", "torch_cpu", "torch_cuda"}:
        raise ValueError("随机状态字段不完整")
    if device.type == "cuda" and len(state["torch_cuda"]) != torch.cuda.device_count():
        raise ValueError("CUDA 随机状态与可见设备数不一致")
    random.setstate(state["python"])
    torch.set_rng_state(state["torch_cpu"].cpu())
    if device.type == "cuda":
        torch.cuda.set_rng_state_all([s.cpu() for s in state["torch_cuda"]])


def cpu_copy(value):
    """state_dict 是浅引用集合；需要独立快照，避免继续训练后连快照也被修改。"""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {k: cpu_copy(v) for k, v in value.items()}
    if isinstance(value, list):
        return [cpu_copy(v) for v in value]
    if isinstance(value, tuple):
        return tuple(cpu_copy(v) for v in value)
    if value is None or type(value) in (bool, int, float, str):
        return value
    raise TypeError(f"拒绝序列化自定义对象：{type(value).__name__}")


def tree_equal(a, b):
    if isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
        return a.dtype == b.dtype and a.shape == b.shape and torch.equal(a.cpu(), b.cpu())
    if type(a) is not type(b):
        return False
    if isinstance(a, dict):
        return a.keys() == b.keys() and all(tree_equal(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)):
        return len(a) == len(b) and all(tree_equal(x, y) for x, y in zip(a, b))
    return a == b


def tensor_diff(a, b):
    if a.keys() != b.keys():
        raise ValueError("模型键不匹配")
    return max(float((a[k].cpu().double()-b[k].cpu().double()).abs().max())
               for k in a if a[k].numel())


def environment(device):
    return {"torch": str(torch.__version__), "python": platform.python_version(),
            "platform": platform.platform(), "device_type": device.type,
            "device_name": (torch.cuda.get_device_name(device) if device.type == "cuda"
                            else platform.machine()),
            "cuda_build": torch.version.cuda,
            "cuda_devices": torch.cuda.device_count() if device.type == "cuda" else 0,
            "threads": torch.get_num_threads(),
            "deterministic": torch.are_deterministic_algorithms_enabled(),
            "deterministic_warn_only": torch.is_deterministic_algorithms_warn_only_enabled(),
            "matmul_precision": torch.get_float32_matmul_precision(),
            "cudnn_benchmark": torch.backends.cudnn.benchmark,
            "cudnn_deterministic": torch.backends.cudnn.deterministic,
            "cudnn_tf32": torch.backends.cudnn.allow_tf32,
            "cublas_workspace": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
            "attention_backend": "SDPA math",
            "implementation_id": IMPLEMENTATION_ID}


@contextmanager
def controlled_environment(device):
    """只在实验期间控制环境，并在退出时恢复。不是跨版本/设备的逐位保证。"""
    old_rng = capture_rng(device)
    old = (torch.get_num_threads(), torch.are_deterministic_algorithms_enabled(),
           torch.is_deterministic_algorithms_warn_only_enabled(),
           torch.get_float32_matmul_precision(), torch.backends.cudnn.benchmark,
           torch.backends.cudnn.deterministic, torch.backends.cudnn.allow_tf32)
    torch.set_num_threads(1)
    torch.use_deterministic_algorithms(True)
    torch.set_float32_matmul_precision("highest")
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.allow_tf32 = False
    try:
        yield
    finally:
        torch.set_num_threads(old[0])
        torch.use_deterministic_algorithms(old[1], warn_only=old[2])
        torch.set_float32_matmul_precision(old[3])
        torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic = old[4:6]
        torch.backends.cudnn.allow_tf32 = old[6]
        restore_rng(old_rng, device)


def autocast_context(precision, device):
    dtype = torch.float16 if precision == "amp_fp16" else torch.bfloat16
    return torch.autocast(device_type=device.type, dtype=dtype, enabled=precision != "fp32")


class CountingAdamW(torch.optim.AdamW):
    """GradScaler 跳步时，这个计数不会增加。自定义属性必须另行保存。"""
    def __init__(self, params, **kwargs):
        super().__init__(params, **kwargs)
        self.actual_steps = 0

    def step(self, closure=None):
        result = super().step(closure=closure)
        self.actual_steps += 1
        return result

In [4]:
# %% 07. 小实验：种子、快照引用与 GradScaler 恢复

def rng_demo():
    state = capture_rng(torch.device("cpu"))
    try:
        torch.manual_seed(26)
        first = torch.rand(4)
        middle_state = torch.get_rng_state().clone()
        expected_next = torch.rand(4)
        torch.manual_seed(26)
        seed_again = torch.rand(4)
        torch.set_rng_state(middle_state)
        resumed_next = torch.rand(4)
        return {"first": first.tolist(), "expected_next": expected_next.tolist(),
                "seed_again": seed_again.tolist(), "resumed_next": resumed_next.tolist(),
                "restored_exact": torch.equal(expected_next, resumed_next),
                "seed_restarts_instead": torch.equal(first, seed_again)}
    finally:
        restore_rng(state, torch.device("cpu"))


def alias_demo():
    layer = nn.Linear(1, 1, bias=False)
    with torch.no_grad():
        layer.weight.fill_(1.0)
    reference = layer.state_dict()
    frozen = cpu_copy(reference)
    with torch.no_grad():
        layer.weight.add_(1.0)
    return {"live_reference": float(reference["weight"].item()),
            "independent_snapshot": float(frozen["weight"].item())}


def scaler_resume_demo():
    """CPU FP32 上验证 GradScaler 控制状态，不声称模拟 CUDA FP16 内核。"""
    def objects():
        p = nn.Parameter(torch.tensor(1.0))
        opt = torch.optim.AdamW([p], lr=0.03, foreach=False)
        sc = torch.amp.GradScaler("cpu", enabled=True, init_scale=128, growth_interval=2)
        return p, opt, sc

    def step(p, opt, sc, inject=False):
        opt.zero_grad(set_to_none=True)
        before = float(p.detach())
        scale_before = sc.get_scale()
        sc.scale((p-3).square()).backward()
        if inject:
            p.grad.fill_(float("inf"))
        sc.step(opt)
        sc.update()
        opt.zero_grad(set_to_none=True)
        return {"value": float(p.detach()), "changed": before != float(p.detach()),
                "scale_before": scale_before, "scale_after": sc.get_scale()}

    p, opt, sc = objects()
    first = [step(p, opt, sc), step(p, opt, sc, inject=True)]
    saved = cpu_copy({"p": p, "optimizer": opt.state_dict(), "scaler": sc.state_dict()})
    expected = [step(p, opt, sc) for _ in range(3)]
    final = cpu_copy({"p": p, "optimizer": opt.state_dict(), "scaler": sc.state_dict()})
    pr, optr, scr = objects()
    with torch.no_grad():
        pr.copy_(saved["p"])
    optr.load_state_dict(saved["optimizer"])
    scr.load_state_dict(saved["scaler"])
    actual = [step(pr, optr, scr) for _ in range(3)]
    final_resume = cpu_copy({"p": pr, "optimizer": optr.state_dict(), "scaler": scr.state_dict()})
    return {"first_two": first, "saved_scaler": saved["scaler"], "continuous_tail": expected,
            "resumed_tail": actual, "all_exact": tree_equal(final, final_resume),
            "fresh_scale": 128.0, "restored_scale": saved["scaler"]["scale"]}

## 2. Seed 与 RNG state：书名不等于书签

把同一随机种子重新设置一遍，会重放序列开头。要接着原来的位置抽样，需要保存生成器当前状态。
`torch.get_rng_state()` 保存的是 CPU 的状态；CUDA 使用单独的接口。Python 的 `random` 也有自己的状态。[S4][S5]

本课只使用标准库 random 与 PyTorch，没有 NumPy 随机操作；真实项目增加 NumPy、第三方增强器或独立生成器时，也要考虑对应状态。

In [5]:
rng_result = rng_demo()
for key, value in rng_result.items():
    print(key, ":", value)

first : [0.8313477635383606, 0.5564764738082886, 0.9642184972763062, 0.49862194061279297]
expected_next : [0.6842003464698792, 0.1909850835800171, 0.04050940275192261, 0.4310958981513977]
seed_again : [0.8313477635383606, 0.5564764738082886, 0.9642184972763062, 0.49862194061279297]
resumed_next : [0.6842003464698792, 0.1909850835800171, 0.04050940275192261, 0.4310958981513977]
restored_exact : True
seed_restarts_instead : True


### 预测题

假设执行顺序是：恢复 RNG → 创建一个随机初始化模型 → 开始训练。
即使随后加载了正确模型权重，Dropout 的随机位置还能完全接上吗？

通常不能：创建模型已经消耗了随机数。**先构造和加载所有对象，最后恢复 RNG，之后直接继续训练。**
不要在两者之间插入随机样本生成、再设置 seed、或创建另一个随机初始化网络。

## 3. `state_dict()` 不是自动冻结的照片

PyTorch 教程提醒，保留 `model.state_dict()` 的结果不等于拿到了独立的参数快照。[S1]
本课对 Tensor 执行 `detach().cpu().clone()`，递归复制容器，才把它用于内存快照。

这个 CPU 拷贝会带来额外内存和同步成本；本课模型很小，优先讲清快照语义。
真实大模型需要另外设计异步、分片或更省内存的保存方式。

In [6]:
print(alias_demo())
# live_reference 随模型从 1 改成 2；independent_snapshot 仍是 1。

{'live_reference': 2.0, 'independent_snapshot': 1.0}


## 4. 数据位置：本课不用“重新开 loader 再跳过若干批”

为了看清原理，本课使用显式、单进程 `DataCursor`，不启用 DataLoader worker、预取或随机增强。
当前轮完整排列与独立 shuffle generator 的状态分开保存。

为什么两者都要保存？生成当前排列时，generator 已经前进了；只保存现在的 generator，
再重新抽一次排列，得到的通常是下一轮排列，而不是当前轮。

多进程 loader 还涉及 worker 随机状态和已经预取但尚未消费的数据。不能把本课的游标恢复，直接当作任意 loader 的完整恢复方案。[S6]

In [7]:
# %% 03. 数据与可恢复游标
def make_data():
    """不同 (a,b) 对只出现一次；重复次数造成长度差异。不是语言能力数据集。"""
    rows = [[BOS] + [a, b]*(1+(a+2*b) % 4) + [EOS]
            for a in range(4, 16) for b in range(4, 16) if a != b]
    order = torch.randperm(len(rows), generator=torch.Generator().manual_seed(2502)).tolist()
    selected = [rows[i] for i in order[:60]]
    return selected[:43], selected[43:]


def collate(rows):
    if not rows or any(len(row) < 2 for row in rows):
        raise ValueError("需要非空样本，且每条至少包含两个 token")
    xs = [torch.tensor(row[:-1], dtype=torch.long) for row in rows]
    ys = [torch.tensor(row[1:], dtype=torch.long) for row in rows]
    x = nn.utils.rnn.pad_sequence(xs, batch_first=True, padding_value=PAD)
    y = nn.utils.rnn.pad_sequence(ys, batch_first=True, padding_value=IGNORE)
    return x, y



def data_signature(train_rows, val_rows):
    # 顺序也属于签名；相同文本集合但行顺序改变，样本编号就可能改变。
    text = json.dumps({"train": train_rows, "validation": val_rows,
                       "special_ids": [PAD, BOS, EOS, IGNORE]},
                      ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


class DataCursor:
    """显式单进程游标。没有 worker、预取或随机数据增强。

    order 是当前 epoch 的完整排列；cursor 指向下一条尚未使用的样本。
    generator 已经越过生成当前 order 的位置，用于下一 epoch 的排列。
    """
    def __init__(self, size: int, seed: int):
        positive_int(size, "size")
        self.size = size
        self.generator = torch.Generator(device="cpu").manual_seed(seed)
        self.epoch, self.cursor = 0, 0
        self.order = torch.randperm(size, generator=self.generator).tolist()

    def take_window(self, micro_size, accumulation_steps):
        positive_int(micro_size, "micro_size")
        positive_int(accumulation_steps, "accumulation_steps")
        if self.cursor == self.size:
            self.epoch += 1
            self.cursor = 0
            self.order = torch.randperm(self.size, generator=self.generator).tolist()
        stop = min(self.cursor + micro_size * accumulation_steps, self.size)
        batches = [self.order[i:min(i+micro_size, stop)]
                   for i in range(self.cursor, stop, micro_size)]
        self.cursor = stop
        return batches

    def state_dict(self):
        return {"size": self.size, "epoch": self.epoch, "cursor": self.cursor,
                "order": self.order.copy(), "generator": self.generator.get_state().clone()}

    def load_state_dict(self, state):
        needed = {"size", "epoch", "cursor", "order", "generator"}
        if not isinstance(state, dict) or set(state) != needed:
            raise ValueError("数据游标字段不完整")
        if state["size"] != self.size:
            raise ValueError("数据长度变化，拒绝伪装为精确恢复")
        if (type(state["epoch"]) is not int or state["epoch"] < 0 or
            type(state["cursor"]) is not int or not 0 <= state["cursor"] <= self.size):
            raise ValueError("epoch/cursor 非法")
        order = state["order"]
        if (not isinstance(order, list) or any(type(i) is not int for i in order)
            or sorted(order) != list(range(self.size))):
            raise ValueError("order 不是完整且无重复的样本排列")
        gen = torch.Generator(device="cpu")
        gen.set_state(state["generator"].cpu())
        self.epoch, self.cursor = state["epoch"], state["cursor"]
        self.order, self.generator = order.copy(), gen

In [8]:
train_rows, val_rows = make_data()
cursor_demo = DataCursor(len(train_rows), seed=2602)
for window in range(4):
    batches = cursor_demo.take_window(3, 4)
    print("窗口", window+1,
          "micro-batch 大小", list(map(len, batches)),
          "已读样本", cursor_demo.cursor,
          "epoch（从0开始）", cursor_demo.epoch)
print("训练/验证条数：", len(train_rows), len(val_rows))

窗口 1 micro-batch 大小 [3, 3, 3, 3] 已读样本 12 epoch（从0开始） 0
窗口 2 micro-batch 大小 [3, 3, 3, 3] 已读样本 24 epoch（从0开始） 0
窗口 3 micro-batch 大小 [3, 3, 3, 3] 已读样本 36 epoch（从0开始） 0
窗口 4 micro-batch 大小 [3, 3, 1] 已读样本 43 epoch（从0开始） 0
训练/验证条数： 43 17


这里一轮有 43 条训练文本：窗口的样本数是 `12、12、12、7`。
最后 7 条会完成一次较小的更新，不能丢掉。进入下一轮时才重新排列。

**要恢复的是“下一条尚未处理的样本”，不是刚刚处理过的那条。**

In [9]:
left = DataCursor(43, seed=2602)
left.take_window(3, 4)
left.take_window(3, 4)
saved_cursor = left.state_dict()
right = DataCursor(43, seed=9999)  # 不同初始 seed 不影响完整状态加载后的结果
right.load_state_dict(saved_cursor)
for i in range(5):
    a = left.take_window(3, 4)
    b = right.take_window(3, 4)
    print("后续窗口", i+1, "样本编号一致：", a == b)
print("数据签名：", data_signature(train_rows, val_rows))

后续窗口 1 样本编号一致： True
后续窗口 2 样本编号一致： True
后续窗口 3 样本编号一致： True
后续窗口 4 样本编号一致： True
后续窗口 5 样本编号一致： True
数据签名： 863f0b756388e903fa8cb8f416c5ed5acb790bda8da2ae8b06bd4be8f7ebe02d


## 5. 什么时候保存？选干净的更新边界

上课的训练目标仍然是每个有效 token 等权。每个小批次的损失和，都除以整个累积窗口的有效 token 数。
只有下面步骤全部结束，才允许保存：

```text
多个 micro-batch 的前向与反向
    → unscale
    → 裁剪梯度
    → scaler.step / optimizer.step
    → scaler.update
    → 如果参数真实更新，scheduler.step
    → zero_grad(set_to_none=True)
    → 更新数据游标和进度，保存
```

本课调度器按**实际成功参数更新次数**推进，不按 micro-batch 次数或尝试次数推进。
GradScaler 跳过更新时，数据和 RNG 已经消耗，因此窗口计数、样本计数仍前进，scheduler 不前进。[S3][S7]

**窗口中途保存不是本课支持的功能。** 真要这样做，还必须处理未提交梯度、它们是否被缩放、窗口分母、剩余小批次和 AMP 的中间状态。更安全的入门方案是从上一个完整窗口重放。

### 支持代码：沿用的模型

两层解码器，52,464 个参数。RoPE、GQA、RMSNorm、SwiGLU 不改变；本课 Dropout 为 0.2。
为了隔离恢复问题，固定使用 SDPA math，单线程与确定性设置用于可重复实验，**不是性能优化建议**。
检查点区域仍保持 `preserve_rng_state=True`，不修改 KV cache 等有状态对象。

In [10]:
# %% 02. 沿用的小型解码器：RoPE / GQA / RMSNorm / SwiGLU
@dataclass(frozen=True)
class ModelConfig:
    vocab_size: int = 16
    d_model: int = 48
    n_heads: int = 4
    n_kv_heads: int = 2
    n_layers: int = 2
    ff_dim: int = 128
    max_len: int = 32
    dropout: float = 0.2
    eps: float = 1e-6

    def __post_init__(self):
        for name in ("vocab_size", "d_model", "n_heads", "n_kv_heads", "n_layers", "ff_dim", "max_len"):
            positive_int(getattr(self, name), name)
        if self.vocab_size < 16 or self.max_len < 9:
            raise ValueError("本课数据要求 vocab_size>=16，max_len>=9")
        if self.d_model % self.n_heads or self.n_heads % self.n_kv_heads:
            raise ValueError("D 必须被 Hq 整除；Hq 必须被 Hkv 整除")
        if (self.d_model // self.n_heads) % 2:
            raise ValueError("RoPE 的每头维度必须为偶数")
        if not (0 <= self.dropout < 1) or not math.isfinite(self.eps) or self.eps <= 0:
            raise ValueError("dropout/eps 非法")


# 沿用已有组件：本课重点是如何恢复训练状态。
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
            z = x.to(dtype)
            z = z * torch.rsqrt(z.square().mean(-1, keepdim=True) + self.eps)
            z = z * self.weight.to(dtype)
        return z.to(x.dtype)


def rope(x, inv_freq):
    with torch.autocast(device_type=x.device.type, enabled=False):
        dtype = torch.float64 if x.dtype == torch.float64 else torch.float32
        pos = torch.arange(x.shape[-2], device=x.device, dtype=dtype)
        angle = pos[:, None] * inv_freq.to(dtype)[None, :]
        c, s = angle.cos()[None, None], angle.sin()[None, None]
        z = x.to(dtype)
        a, b = z[..., 0::2], z[..., 1::2]
        out = torch.stack((a*c-b*s, a*s+b*c), -1).flatten(-2)
    return out.to(x.dtype)


class Attention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg, self.d = cfg, cfg.d_model // cfg.n_heads
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.d, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.n_kv_heads*self.d, bias=False)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.register_buffer("inv_freq", 10000.0 ** (-torch.arange(0, self.d, 2).float()/self.d))

    def forward(self, x, allowed):
        b, t, _ = x.shape
        q = self.q(x).reshape(b, t, self.cfg.n_heads, self.d).transpose(1, 2)
        k = self.k(x).reshape(b, t, self.cfg.n_kv_heads, self.d).transpose(1, 2)
        v = self.v(x).reshape(b, t, self.cfg.n_kv_heads, self.d).transpose(1, 2)
        q, k = rope(q, self.inv_freq), rope(k, self.inv_freq)
        # 显式 mask：True 允许读取；pad query 保留可访问 BOS，避免全屏蔽行。
        with sdpa_kernel(SDPBackend.MATH):
            z = F.scaled_dot_product_attention(
                q, k, v, attn_mask=allowed, dropout_p=0.0,
                enable_gqa=self.cfg.n_heads != self.cfg.n_kv_heads,
            )
        return self.out(z.transpose(1, 2).reshape(b, t, -1))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.norm2 = RMSNorm(cfg.d_model, cfg.eps), RMSNorm(cfg.d_model, cfg.eps)
        self.attn = Attention(cfg)
        self.gate = nn.Linear(cfg.d_model, cfg.ff_dim, bias=False)
        self.up = nn.Linear(cfg.d_model, cfg.ff_dim, bias=False)
        self.down = nn.Linear(cfg.ff_dim, cfg.d_model, bias=False)
        self.drop1, self.drop2 = nn.Dropout(cfg.dropout), nn.Dropout(cfg.dropout)

    def forward(self, x, allowed):
        x = x + self.drop1(self.attn(self.norm1(x), allowed))
        h = self.norm2(x)
        return x + self.drop2(self.down(F.silu(self.gate(h)) * self.up(h)))


class TinyDecoder(nn.Module):
    def __init__(self, cfg=ModelConfig(), activation_checkpointing=False, preserve_rng_state=True):
        super().__init__()
        self.cfg = cfg
        self.activation_checkpointing = activation_checkpointing
        self.preserve_rng_state = preserve_rng_state
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d_model, cfg.eps)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

    def forward(self, ids):
        if ids.dtype != torch.long or ids.ndim != 2 or min(ids.shape) == 0:
            raise ValueError("ids 必须是非空 long [B,T]")
        if ids.shape[1] > self.cfg.max_len:
            raise ValueError("超过最大长度")
        if bool((ids < 0).any()) or bool((ids >= self.cfg.vocab_size).any()):
            raise ValueError("token 编号越界")
        if bool((ids[:, 0] == PAD).any()) or bool(((ids[:, :-1] == PAD) & (ids[:, 1:] != PAD)).any()):
            raise ValueError("本课只支持非空、右侧补齐的序列")
        t = ids.shape[1]
        causal = torch.ones(t, t, device=ids.device, dtype=torch.bool).tril()
        allowed = causal[None, None] & (ids != PAD)[:, None, None, :]
        x = self.embedding(ids)
        for block in self.blocks:
            if self.activation_checkpointing and self.training and torch.is_grad_enabled():
                # 直接传当前 block，不使用捕获循环变量的 lambda。
                # 没有更新 KV cache、运行统计或其他影响结果的副作用。
                x = checkpoint(block, x, allowed, use_reentrant=False,
                               preserve_rng_state=self.preserve_rng_state)
            else:
                x = block(x, allowed)
        return self.head(self.norm(x))

In [11]:
# %% 05. 窗口边界训练器
class Trainer:
    def __init__(self, model_cfg=ModelConfig(), train_cfg=TrainConfig(),
                 device=torch.device("cpu"), data=None):
        if train_cfg.precision == "amp_fp16" and device.type != "cuda":
            raise ValueError("本课完整 FP16 训练只在 CUDA 路径启用")
        if (train_cfg.precision == "amp_bf16" and device.type == "cuda" and
                not torch.cuda.is_bf16_supported()):
            raise RuntimeError("当前 CUDA 环境未报告 BF16 支持")
        self.model_cfg, self.cfg, self.device = model_cfg, train_cfg, device
        self.train_rows, self.val_rows = copy.deepcopy(make_data() if data is None else data)
        self.data_hash = data_signature(self.train_rows, self.val_rows)
        seed_all(train_cfg.seed)
        self.model = TinyDecoder(model_cfg, train_cfg.activation_checkpointing).to(device)
        self.optimizer = CountingAdamW(self.model.parameters(), lr=train_cfg.lr,
                                       weight_decay=train_cfg.weight_decay, foreach=False)
        # 先构造 scheduler，随后恢复时再加载状态，避免初始化覆盖恢复的 lr。
        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=train_cfg.scheduler_step, gamma=train_cfg.scheduler_gamma)
        self.scaler = torch.amp.GradScaler(device.type, enabled=train_cfg.precision == "amp_fp16",
                                          init_scale=128.0, growth_interval=4)
        self.cursor = DataCursor(len(self.train_rows), train_cfg.seed+1)
        self.progress = {"windows_done": 0, "updates_done": 0,
                         "samples_seen": 0, "tokens_seen": 0}
        self.history = []
        self.at_boundary = True
        self.optimizer.zero_grad(set_to_none=True)

    def update_one_window(self, inject_inf=False):
        if not self.at_boundary:
            raise RuntimeError("上次窗口未完成；请从最近的完整检查点恢复，而不是继续不完整状态")
        self.at_boundary = False
        ids_batches = self.cursor.take_window(self.cfg.micro_size, self.cfg.accumulation_steps)
        batches = [collate([self.train_rows[i] for i in ids]) for ids in ids_batches]
        token_count = sum(int((y != IGNORE).sum()) for _, y in batches)
        self.model.train()
        self.optimizer.zero_grad(set_to_none=True)
        lr_used = float(self.optimizer.param_groups[0]["lr"])
        before = self.optimizer.actual_steps
        scale_before = self.scaler.get_scale()
        summed_loss = 0.0
        for x_cpu, y_cpu in batches:
            x, y = x_cpu.to(self.device), y_cpu.to(self.device)
            with autocast_context(self.cfg.precision, self.device):
                logits = self.model(x)
                loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1),
                                       ignore_index=IGNORE, reduction="sum")
            if not bool(torch.isfinite(loss)):
                raise FloatingPointError("前向损失不是有限数；本窗口不允许保存")
            self.scaler.scale(loss / token_count).backward()
            summed_loss += float(loss.detach())
        if inject_inf:
            if not self.scaler.is_enabled():
                raise ValueError("故障注入要求启用 GradScaler")
            next(p for p in self.model.parameters() if p.grad is not None).grad.fill_(float("inf"))
        self.scaler.unscale_(self.optimizer)
        grad_norm = nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.max_norm,
                                             error_if_nonfinite=not self.scaler.is_enabled())
        self.scaler.step(self.optimizer)
        self.scaler.update()
        updated = self.optimizer.actual_steps > before
        # 本课 scheduler 的时钟是成功参数更新，不是 micro-batch 或尝试次数。
        if updated:
            self.scheduler.step()
        self.optimizer.zero_grad(set_to_none=True)
        self.progress["windows_done"] += 1
        self.progress["updates_done"] = self.optimizer.actual_steps
        self.progress["samples_seen"] += sum(map(len, ids_batches))
        self.progress["tokens_seen"] += token_count
        record = {"window": self.progress["windows_done"], "epoch": self.cursor.epoch,
                  "cursor": self.cursor.cursor, "sample_ids": [i for ids in ids_batches for i in ids],
                  "micro_sizes": list(map(len, ids_batches)), "valid_tokens": token_count,
                  "loss": summed_loss/token_count, "lr_used": lr_used,
                  "lr_next": float(self.optimizer.param_groups[0]["lr"]),
                  "grad_norm": float(grad_norm) if bool(torch.isfinite(grad_norm)) else None,
                  "scale_before": scale_before, "scale_after": self.scaler.get_scale(),
                  "updated": updated}
        self.history.append(record)
        self.at_boundary = True
        return record

    def train_to(self, total_windows, on_boundary=None):
        positive_int(total_windows, "total_windows")
        if total_windows < self.progress["windows_done"]:
            raise ValueError("目标窗口数小于已完成窗口数；total_windows 是累计总数，不是新增数")
        while self.progress["windows_done"] < total_windows:
            record = self.update_one_window()
            if on_boundary is not None:
                on_boundary(self, record)
        return self

    def snapshot(self):
        if not self.at_boundary or any(p.grad is not None for p in self.model.parameters()):
            raise RuntimeError("只能保存完整窗口边界：step/scaler/scheduler 已结束，所有 grad 必须为 None")
        return cpu_copy({
            "format_version": FORMAT_VERSION, "implementation_id": IMPLEMENTATION_ID,
            "model_config": asdict(self.model_cfg), "train_config": asdict(self.cfg),
            "environment": environment(self.device), "data_signature": self.data_hash,
            "parameter_names": [name for name, _ in self.model.named_parameters()],
            "model": self.model.state_dict(), "model_training": self.model.training,
            "optimizer": self.optimizer.state_dict(), "scheduler": self.scheduler.state_dict(),
            "scaler": self.scaler.state_dict(), "rng": capture_rng(self.device),
            "data_cursor": self.cursor.state_dict(), "progress": self.progress,
            "history": self.history, "boundary": "after_window_grad_none",
        })

    @torch.no_grad()
    def evaluate(self):
        was_training = self.model.training
        self.model.eval()
        total, count = 0.0, 0
        try:
            for start in range(0, len(self.val_rows), 8):
                x, y = collate(self.val_rows[start:start+8])
                x, y = x.to(self.device), y.to(self.device)
                # 所有分支统一用 FP32 模型前向验证，不改变训练 RNG。
                with torch.autocast(device_type=self.device.type, enabled=False):
                    logits = self.model(x)
                    loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), y.reshape(-1),
                                           ignore_index=IGNORE, reduction="sum")
                total += float(loss)
                count += int((y != IGNORE).sum())
            return total/count
        finally:
            self.model.train(was_training)

### 学习率调度器的新概念

本课 `StepLR(step_size=3, gamma=0.8)` 在每三次**成功更新**后将学习率乘以 0.8。
名称里的 `last_epoch` 在本课表示调度器已被推进的次数，不是数据的 epoch 编号。
恢复时不要只抄一个 lr 数字，还要恢复后续何时衰减的状态。

下面打印 `lr_used`（本次更新用的值）与 `lr_next`（之后的值），避免两种日志口径混淆。

In [12]:
with controlled_environment(torch.device("cpu")):
    lr_probe = Trainer(model_cfg=ModelConfig(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32))
    for _ in range(4):
        row = lr_probe.update_one_window()
        print("窗口", row["window"], "lr_used", row["lr_used"],
              "lr_next", row["lr_next"], "真实更新", row["updated"])
    print("窗口后所有 grad 都为 None：",
          all(p.grad is None for p in lr_probe.model.parameters()))

窗口 1 lr_used 0.003 lr_next 0.003 真实更新 True
窗口 2 lr_used 0.003 lr_next 0.003 真实更新 True
窗口 3 lr_used 0.003 lr_next 0.0024000000000000002 真实更新 True
窗口 4 lr_used 0.0024000000000000002 lr_next 0.0024000000000000002 真实更新 True
窗口后所有 grad 都为 None： True


## 6. 恢复顺序：对象先建好，RNG 最后接回

本课按以下顺序：

```text
检查版本、环境、数据签名
    → 重建 model、optimizer、scheduler、scaler
    → model.load_state_dict(strict=True)
    → scheduler.load_state_dict
    → optimizer.load_state_dict
    → scaler、数据游标、计数、模式
    → 清空梯度
    → 最后恢复 RNG
    → 下一次训练
```

关键是**加载 optimizer 之前，scheduler 已经完成构造**，避免构造过程覆盖已恢复的学习率。[S2]
本课还核对模型参数名字的注册顺序，防止相同数量的参数被错误地配到优化器状态上。

恢复会拒绝旧课的仅权重检查点、数据变化、参数顺序变化或本课环境签名不一致。
固定 `IMPLEMENTATION_ID` 是手动维护的代码版本标识，不会自动检测你任意修改过的 Python 源码。

In [13]:
# %% 06. 恢复、校验与原子替换
REQUIRED_KEYS = {"format_version", "implementation_id", "model_config", "train_config",
                 "environment", "data_signature", "parameter_names", "model", "model_training",
                 "optimizer", "scheduler", "scaler", "rng", "data_cursor", "progress",
                 "history", "boundary"}


def validate_payload(payload, device, data=None):
    if not isinstance(payload, dict) or set(payload) != REQUIRED_KEYS:
        raise ValueError("不是本课完整训练检查点；旧课仅权重文件不能精确续训")
    if payload["format_version"] != FORMAT_VERSION or payload["implementation_id"] != IMPLEMENTATION_ID:
        raise ValueError("检查点格式/实现版本不兼容")
    if payload["boundary"] != "after_window_grad_none":
        raise ValueError("不支持窗口中途保存的检查点")
    if payload["environment"] != environment(device):
        raise ValueError("运行环境与检查点不一致；本课严格恢复拒绝跨环境。迁移模型做推理是另一种操作。")
    expected = data_signature(*(make_data() if data is None else data))
    if payload["data_signature"] != expected:
        raise ValueError("数据内容或顺序已改变，拒绝恢复")
    mc, tc = ModelConfig(**payload["model_config"]), TrainConfig(**payload["train_config"])
    progress = payload["progress"]
    if set(progress) != {"windows_done", "updates_done", "samples_seen", "tokens_seen"}:
        raise ValueError("进度字段不完整")
    if any(type(v) is not int or v < 0 for v in progress.values()):
        raise ValueError("进度计数必须为非负整数")
    if progress["updates_done"] > progress["windows_done"]:
        raise ValueError("成功更新数不可能超过尝试窗口数")
    if len(payload["history"]) != progress["windows_done"]:
        raise ValueError("历史记录长度与进度不符")
    if payload["scheduler"].get("last_epoch") != progress["updates_done"]:
        raise ValueError("scheduler 时钟与成功更新数不符")
    if bool(payload["scaler"]) != (tc.precision == "amp_fp16"):
        raise ValueError("scaler 状态与精度配置不符")
    return mc, tc


def restore_trainer(payload, device=torch.device("cpu"), data=None, _fault=None):
    """公开用法不传 _fault；它仅用于本课故障注入，不是恢复选项。

    所有对象先构造，再加载 model/scheduler/optimizer/scaler/data，最后恢复 RNG。
    """
    mc, tc = validate_payload(payload, device, data)
    if _fault not in (None, "optimizer", "rng", "cursor", "scheduler"):
        raise ValueError("未知故障名")
    trainer = Trainer(mc, tc, device, data)
    if [name for name, _ in trainer.model.named_parameters()] != payload["parameter_names"]:
        raise ValueError("优化器参数注册顺序不匹配")
    trainer.model.load_state_dict(payload["model"], strict=True)
    trainer.model.train(payload["model_training"])
    # scheduler 已构造；先加载其状态，再加载 optimizer 恢复学习率和动量。
    if _fault != "scheduler":
        trainer.scheduler.load_state_dict(copy.deepcopy(payload["scheduler"]))
    if _fault != "optimizer":
        trainer.optimizer.load_state_dict(copy.deepcopy(payload["optimizer"]))
    else:
        # 隔离“缺少 Adam 状态”影响：保留正确 lr，不顺便重置学习率。
        for group, saved in zip(trainer.optimizer.param_groups, payload["optimizer"]["param_groups"]):
            for key, value in saved.items():
                if key != "params":
                    group[key] = copy.deepcopy(value)
    trainer.scaler.load_state_dict(copy.deepcopy(payload["scaler"]))
    trainer.cursor.load_state_dict(payload["data_cursor"])
    trainer.progress = copy.deepcopy(payload["progress"])
    trainer.history = copy.deepcopy(payload["history"])
    trainer.optimizer.actual_steps = trainer.progress["updates_done"]
    trainer.optimizer.zero_grad(set_to_none=True)
    if _fault == "cursor":
        trainer.cursor.cursor = 0  # 错误地从当前 epoch 第一条重新开始
    elif (trainer.cursor.cursor != trainer.cursor.size and
          trainer.cursor.cursor % (tc.micro_size*tc.accumulation_steps) != 0):
        raise ValueError("cursor 不在完整累积窗口边界")
    # 从此处到下一次训练，不再抽样、不再重设种子、不再构造随机参数模型。
    if _fault != "rng":
        restore_rng(payload["rng"], device)
    return trainer


def atomic_torch_save(payload, path, save_fn=None):
    """同目录临时文件 -> flush/fsync -> os.replace。不是断电持久性或多写者协议。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    writer = torch.save if save_fn is None else save_fn
    temp_path = None
    try:
        with tempfile.NamedTemporaryFile(mode="w+b", prefix=f".{path.name}.", suffix=".tmp",
                                         dir=path.parent, delete=False) as file:
            temp_path = Path(file.name)
            writer(payload, file)
            file.flush()
            os.fsync(file.fileno())
        os.replace(temp_path, path)
    finally:
        if temp_path is not None and temp_path.exists():
            temp_path.unlink()


def load_checkpoint(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"找不到检查点：{path}")
    # CPU 加载让 RNG ByteTensor 保持在 CPU；optimizer.load_state_dict 负责参数状态迁移。
    # weights_only 不是“只取 model”开关，基础容器中的 optimizer/RNG 等仍会读到。
    return torch.load(path, map_location="cpu", weights_only=True)


def save_checkpoint(trainer, path):
    atomic_torch_save(trainer.snapshot(), path)


def compare_snapshots(a, b):
    parts = ("model", "optimizer", "scheduler", "scaler", "rng", "data_cursor", "progress", "history")
    result = {name+"_exact": tree_equal(a[name], b[name]) for name in parts}
    result["model_max_abs_difference"] = tensor_diff(a["model"], b["model"])
    result["all_exact"] = all(result[name+"_exact"] for name in parts)
    return result


def write_json(data, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

## 7. 磁盘保存：不要先毁掉唯一一份可恢复文件

本课先在目标目录写一个临时文件，再 `flush()`、`fsync()`，完成后用 `os.replace()` 替换最终路径。
成功的同文件系统替换可避免读者看到“正在写到一半”的最终文件。[S8]

这不等于所有文件系统上的断电持久性保证，也不解决多个写入者互相覆盖。
本课是单写入者、同步保存，并清理写入失败留下的临时文件。

下面故意在临时文件里只写几个字节，然后抛异常。旧文件必须保持不变。

In [14]:
with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "latest.pt"
    atomic_torch_save({"finished_window": 3}, path)
    before = path.read_bytes()

    def broken_writer(payload, file):
        file.write(b"incomplete")
        raise OSError("模拟磁盘写入失败")

    try:
        atomic_torch_save({"finished_window": 4}, path, save_fn=broken_writer)
    except OSError as error:
        print("捕获：", error)
    print("旧文件保持不变：", path.read_bytes() == before)
    print("旧文件仍可读取：", load_checkpoint(path))

捕获： 模拟磁盘写入失败
旧文件保持不变： True
旧文件仍可读取： {'finished_window': 3}


### 安全加载与 `weights_only=True`

本课显式使用：

```python
payload = torch.load(path, map_location="cpu", weights_only=True)
```

`weights_only=True` **不是只读取 `model` 字段**。基础字典、列表、元组和 Tensor 中的优化器、RNG 等状态也能加载。
它限制反序列化能创建的对象/调用的函数，而不是筛选字典键。

本课不回退到 `weights_only=False`，不随意放行自定义类。只打开可信来源的文件；这一设置缩小攻击面，但不防所有资源耗尽或内存风险。[S9]

数据 SHA-256 签名只帮助发现本课数据不同，并不是对来源的认证，也不是完整检查点的数字签名。

## 8. AMP 的 scaler 也有记忆

保存 GradScaler 应在该次 `update()` 完成以后，或下一次前向开始以前。[S3]
仅创建一个新的 scaler，会丢掉已调整的 scale 和增长计数。

下面在 **CPU 的 FP32 标量**上使用真正的 GradScaler，验证控制状态；不是对 CUDA FP16 内核的替代测量。
第二次故意注入非有限梯度，观察参数不更新、scale 降低，再从该位置恢复。

In [15]:
scaler_demo = scaler_resume_demo()
print("前两次：", json.dumps(scaler_demo["first_two"], ensure_ascii=False, indent=2))
print("保存的 scaler：", scaler_demo["saved_scaler"])
print("新建 scale / 恢复 scale：", scaler_demo["fresh_scale"], scaler_demo["restored_scale"])
print("恢复后的参数、优化器与 scaler 一致：", scaler_demo["all_exact"])
print("恢复后的逐步记录一致：", scaler_demo["continuous_tail"] == scaler_demo["resumed_tail"])

前两次： [
  {
    "value": 1.0297000408172607,
    "changed": true,
    "scale_before": 128,
    "scale_after": 128.0
  },
  {
    "value": 1.0297000408172607,
    "changed": false,
    "scale_before": 128.0,
    "scale_after": 64.0
  }
]
保存的 scaler： {'scale': 64.0, 'growth_factor': 2.0, 'backoff_factor': 0.5, 'growth_interval': 2, '_growth_tracker': 0}
新建 scale / 恢复 scale： 128.0 64.0
恢复后的参数、优化器与 scaler 一致： True
恢复后的逐步记录一致： True


## 9. 核心实验：连续训练 vs. 真正接着训练

默认连续完成 16 个累积窗口；另一条路径完成 7 个窗口后保存、删除对象、扰动 RNG、重新创建全部对象，再恢复到第 16 个窗口。

第 7 个窗口结束时处于**第二个数据 epoch 中途**，不是恰好停在 epoch 末尾。这样更容易发现游标错误。
每轮窗口样本数为 `12、12、12、7`，暂停点下一窗口应该消费剩余的 7 条，而不是重读已处理的 36 条。

比较的不只是最终 loss，还包括模型、优化器、调度器、scaler、RNG、数据游标、计数、完整轨迹。
这里记录的 loss 是训练工程验证，不是模型能力排名；本课不选择“最佳轮次”。

In [16]:
# %% 08. 连续 / 恢复 / 故障对照

def run_experiment(output_dir, device=torch.device("cpu"), precision="fp32",
                   total_windows=16, interrupt_after=7, include_faults=True):
    positive_int(total_windows, "total_windows")
    if type(interrupt_after) is not int or not 0 < interrupt_after < total_windows:
        raise ValueError("interrupt_after 必须位于 0 与 total_windows 之间")
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    cfg = TrainConfig(precision=precision)
    continuous = Trainer(train_cfg=cfg, device=device)
    initial_loss = continuous.evaluate()
    continuous.train_to(total_windows)
    continuous_loss = continuous.evaluate()
    full = continuous.snapshot()
    atomic_torch_save(full, out/"continuous.pt")
    del continuous

    interrupted = Trainer(train_cfg=cfg, device=device)
    interrupted.train_to(interrupt_after)
    save_checkpoint(interrupted, out/"interrupt.pt")
    cut = load_checkpoint(out/"interrupt.pt")
    del interrupted
    # 模拟退出后的无关 RNG 消耗。恢复必须覆盖它，而不能依赖当前进程恰好未消耗随机数。
    torch.rand(37)
    random.random()
    resumed = restore_trainer(cut, device)
    resumed.train_to(total_windows)
    resumed_loss = resumed.evaluate()
    final = resumed.snapshot()
    atomic_torch_save(final, out/"resumed.pt")
    comparison = compare_snapshots(full, final)
    if not comparison["all_exact"]:
        raise RuntimeError(f"本环境精确恢复检查失败，不能宣称完全等价：{comparison}")
    del resumed

    faults = {}
    if include_faults:
        for fault in ("optimizer", "rng", "cursor", "scheduler"):
            bad = restore_trainer(cut, device, _fault=fault)
            bad.train_to(total_windows)
            bad_loss = bad.evaluate()
            # 故障路径的 scheduler 状态可能故意不一致，因此只作实验比较，不写成可恢复文件。
            bad_state = bad.snapshot()
            first_ids = next((i+1 for i, (x, y) in enumerate(zip(full["history"], bad.history))
                              if x["sample_ids"] != y["sample_ids"]), None)
            first_lr = next((i+1 for i, (x, y) in enumerate(zip(full["history"], bad.history))
                             if x["lr_used"] != y["lr_used"]), None)
            first_loss = next((i+1 for i, (x, y) in enumerate(zip(full["history"], bad.history))
                               if x["loss"] != y["loss"]), None)
            faults[fault] = {"model_max_abs_difference": tensor_diff(full["model"], bad_state["model"]),
                             "validation_loss": bad_loss,
                             "first_different_sample_window": first_ids,
                             "first_different_lr_window": first_lr,
                             "first_different_loss_window": first_loss}
            if faults[fault]["model_max_abs_difference"] == 0:
                raise AssertionError(f"故障 {fault} 未被这个实验检测到，请调整实验而不是隐瞒")
            del bad
    report = {"environment": full["environment"], "model_config": full["model_config"],
              "train_config": full["train_config"],
              "parameter_count": sum(v.numel() for k, v in full["model"].items() if not k.endswith("inv_freq")),
              "data": {"train_samples": 43, "validation_samples": 17,
                       "signature": full["data_signature"], "random_augmentation": False,
                       "worker_processes": 0},
              "total_windows": total_windows, "interrupt_after": interrupt_after,
              "interrupt_cursor": {k: v for k, v in cut["data_cursor"].items() if k != "generator"},
              "initial_validation_loss": initial_loss,
              "continuous_validation_loss": continuous_loss,
              "resumed_validation_loss": resumed_loss,
              "comparison": comparison, "faults": faults,
              "boundaries": "仅完整累积窗口；无跨环境、跨设备或多进程 loader 的精确保证",
              "cpu_cuda_test_note": "报告仅对应 environment 中的设备，不是 GPU 速度或语言能力评测"}
    write_json(report, out/"experiment.json")
    write_json({"continuous": full["history"], "resumed": final["history"]}, out/"history.json")
    return report

In [17]:
with controlled_environment(DEVICE):
    report = run_experiment(
        OUTPUT_DIR, device=DEVICE, precision=PRECISION,
        total_windows=TOTAL_WINDOWS, interrupt_after=INTERRUPT_AFTER,
        include_faults=True,
    )
print("参数量：", report["parameter_count"])
print("暂停位置：epoch（从0开始）", report["interrupt_cursor"]["epoch"],
      "下一样本游标", report["interrupt_cursor"]["cursor"])
print("初始 / 连续最终 / 恢复最终验证损失：",
      report["initial_validation_loss"], report["continuous_validation_loss"],
      report["resumed_validation_loss"])

参数量： 52464
暂停位置：epoch（从0开始） 1 下一样本游标 36
初始 / 连续最终 / 恢复最终验证损失： 2.854015545178485 2.6489115889354418 2.6489115889354418


In [18]:
print(json.dumps(report["comparison"], ensure_ascii=False, indent=2))
print("注意：这是报告中这个软硬件环境的实测，不是跨设备/跨版本保证。")

{
  "model_exact": true,
  "optimizer_exact": true,
  "scheduler_exact": true,
  "scaler_exact": true,
  "rng_exact": true,
  "data_cursor_exact": true,
  "progress_exact": true,
  "history_exact": true,
  "model_max_abs_difference": 0.0,
  "all_exact": true
}
注意：这是报告中这个软硬件环境的实测，不是跨设备/跨版本保证。


### 四组故障对照：一次只漏一类状态

- `optimizer`：保留正确学习率，但丢掉 Adam 动量与步数；
- `rng`：保留模型/优化器/数据顺序，却让 Dropout 从新对象初始化后的随机位置继续；
- `cursor`：保留当前轮排列，却从该轮第 0 条样本重放；
- `scheduler`：保留当前学习率，却丢掉下一次衰减的阶段。

这些只是教学故障分支，不是提供给实际训练的“可忽略字段”选项。

In [19]:
for fault, values in report["faults"].items():
    print(fault,
          "参数最大差异：", values["model_max_abs_difference"],
          "首次样本不同窗口：", values["first_different_sample_window"],
          "首次学习率不同窗口：", values["first_different_lr_window"],
          "首次 loss 不同窗口：", values["first_different_loss_window"])

optimizer 参数最大差异： 0.015056326985359192 首次样本不同窗口： None 首次学习率不同窗口： None 首次 loss 不同窗口： 9
rng 参数最大差异： 0.010317627340555191 首次样本不同窗口： None 首次学习率不同窗口： None 首次 loss 不同窗口： 8
cursor 参数最大差异： 0.011708037927746773 首次样本不同窗口： 8 首次学习率不同窗口： None 首次 loss 不同窗口： 8
scheduler 参数最大差异： 0.0009440183639526367 首次样本不同窗口： None 首次学习率不同窗口： 10 首次 loss 不同窗口： 11


### 反直觉观察：错误可能迟一个窗口才在 loss 上出现

漏掉 optimizer 时，恢复后第一次前向的权重、数据与 Dropout 仍正确，因此这次的更新前 loss 可以相同。
但该次参数更新已经用错动量，下一个窗口的 loss 才可能明显不同。

所以不能只比较恢复后的第一条 loss 日志。**同时检查状态与后续轨迹。**

In [20]:
saved = load_checkpoint(OUTPUT_DIR / "interrupt.pt")
print("检查点键：", sorted(saved.keys()))
print("Adam 状态项数：", len(saved["optimizer"]["state"]))
print("进度：", saved["progress"])
print("下一次学习率：", saved["optimizer"]["param_groups"][0]["lr"])
print("调度器状态：", saved["scheduler"])
print("文件大小（字节）：", (OUTPUT_DIR / "interrupt.pt").stat().st_size)
# 不把 .pt 压缩文件字节相等当作状态等价的唯一标准；本课比较其中的实际数据。

检查点键： ['boundary', 'data_cursor', 'data_signature', 'environment', 'format_version', 'history', 'implementation_id', 'model', 'model_config', 'model_training', 'optimizer', 'parameter_names', 'progress', 'rng', 'scaler', 'scheduler', 'train_config']
Adam 状态项数： 21
进度： {'windows_done': 7, 'updates_done': 7, 'samples_seen': 79, 'tokens_seen': 489}
下一次学习率： 0.0019200000000000003
调度器状态： {'step_size': 3, 'gamma': 0.8, 'base_lrs': [0.003], 'last_epoch': 7, '_step_count': 8, '_is_initial': False, '_get_lr_called_within_step': False, '_last_lr': [0.0019200000000000003]}
文件大小（字节）： 672155


## 10. 自动检查与真正的跨进程复现

先运行下列固定检查。它们使用更小的解码器，覆盖正确恢复、拒绝非法检查点、失败保存保留旧文件、随机状态、尾窗口与故障分支。

**检查在 CPU 上运行。** 它们不能替代你选择 CUDA 后的完整训练比较。

In [21]:
# %% 09. 自动检查：实际计数，不使用 assert 处理外部输入
class Checks:
    def __init__(self):
        self.names = []

    def check(self, condition, name):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def raises(self, exc_type, fn, name):
        try:
            fn()
        except exc_type:
            self.names.append(name)
        else:
            raise AssertionError(name+"：未抛出预期异常")


def run_checks():
    c = Checks()
    dev = torch.device("cpu")
    with controlled_environment(dev):
        demo = rng_demo()
        c.check(demo["restored_exact"], "恢复 RNG 状态会接着抽样")
        c.check(demo["seed_restarts_instead"], "重新设 seed 会回到序列起点")
        c.check(demo["expected_next"] != demo["seed_again"], "seed 不等于中途状态")
        rng = capture_rng(dev)
        p1, t1 = random.random(), torch.rand(5)
        restore_rng(rng, dev)
        c.check(random.random() == p1, "Python RNG 恢复")
        c.check(torch.equal(torch.rand(5), t1), "CPU torch RNG 恢复")
        alias = alias_demo()
        c.check(alias == {"live_reference": 2.0, "independent_snapshot": 1.0}, "快照不能保留活引用")
        tr, va = make_data()
        c.check((len(tr), len(va)) == (43, 17), "数据长度")
        c.check(data_signature(tr, va) != data_signature(tr[::-1], va), "数据签名包括行顺序")
        s = DataCursor(43, 99)
        batch_sizes = [sum(map(len, s.take_window(3, 4))) for _ in range(4)]
        c.check(batch_sizes == [12, 12, 12, 7], "保留不足一个窗口的尾样本")
        c.check(s.cursor == 43 and s.epoch == 0, "epoch 结束边界可保存")
        saved = s.state_dict()
        s2 = DataCursor(43, 777)
        s2.load_state_dict(saved)
        c.check(tree_equal(s.state_dict(), s2.state_dict()), "数据游标完整恢复")
        for i in range(6):
            c.check(s.take_window(3, 4) == s2.take_window(3, 4), f"数据顺序跨 epoch 恢复 {i+1}")
        bad_order = cpu_copy(saved); bad_order["order"][0] = bad_order["order"][1]
        c.raises(ValueError, lambda: s2.load_state_dict(bad_order), "拒绝重复样本编号")
        bad_cursor = cpu_copy(saved); bad_cursor["cursor"] = 44
        c.raises(ValueError, lambda: s2.load_state_dict(bad_cursor), "拒绝越界 cursor")
        c.raises(ValueError, lambda: DataCursor(0, 1), "拒绝空训练数据")
        c.raises(ValueError, lambda: s.take_window(0, 4), "拒绝零 micro-batch")

        # 使用较小但完整的两层解码器，使快速检查不变成完整训练。
        mc = ModelConfig(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32, dropout=0.25)
        cfg = TrainConfig()
        a = Trainer(mc, cfg, dev)
        c.check(all(p.grad is None for p in a.model.parameters()), "初始边界无残留梯度")
        a.train_to(5)
        cut = a.snapshot()
        frozen_model = cpu_copy(cut["model"])
        a.train_to(9)
        full = a.snapshot()
        c.check(tree_equal(cut["model"], frozen_model), "继续训练不改动内存快照")
        a.evaluate()
        c.check(tree_equal(full["rng"], capture_rng(dev)), "验证不消耗训练 RNG")
        c.check(a.model.training, "验证后恢复 train 模式")
        b = restore_trainer(cut, dev)
        c.check(b.progress == cut["progress"], "恢复进度")
        c.check(b.optimizer.actual_steps == cut["progress"]["updates_done"], "恢复自定义真实更新计数")
        c.check(tree_equal(b.snapshot()["optimizer"], cut["optimizer"]), "恢复 Adam 动量与步数")
        c.check(b.scheduler.state_dict() == cut["scheduler"], "恢复 scheduler")
        c.check(b.scaler.state_dict() == cut["scaler"], "恢复 disabled scaler 空状态")
        c.check(all(p.grad is None for p in b.model.parameters()), "恢复后 grad=None")
        c.check(b.model.training, "恢复模型模式")
        c.check(tree_equal(capture_rng(dev), cut["rng"]), "RNG 最后恢复而不再被初始化消耗")
        b.train_to(9)
        resumed = b.snapshot()
        for part in ("model", "optimizer", "scheduler", "scaler", "rng", "data_cursor", "progress", "history"):
            c.check(tree_equal(full[part], resumed[part]), f"连续/恢复逐位一致：{part}")
        c.check(compare_snapshots(full, resumed)["all_exact"], "恢复总检查")
        c.raises(ValueError, lambda: b.train_to(8), "累计总窗口数不得倒退")
        b.at_boundary = False
        c.raises(RuntimeError, b.snapshot, "拒绝窗口中途保存")
        b.at_boundary = True
        next(b.model.parameters()).grad = torch.zeros_like(next(b.model.parameters()))
        c.raises(RuntimeError, b.snapshot, "拒绝未清理的梯度")
        b.optimizer.zero_grad(set_to_none=True)
        c.check(b.snapshot()["boundary"] == "after_window_grad_none", "完整边界标记")

        for key in ("model", "optimizer", "rng", "data_cursor", "scaler"):
            bad = cpu_copy(cut); del bad[key]
            c.raises(ValueError, lambda bad=bad: restore_trainer(bad, dev), f"拒绝缺字段：{key}")
        bad = cpu_copy(cut); bad["format_version"] = 100
        c.raises(ValueError, lambda: restore_trainer(bad, dev), "拒绝格式版本不兼容")
        bad = cpu_copy(cut); bad["environment"]["torch"] = "different"
        c.raises(ValueError, lambda: restore_trainer(bad, dev), "拒绝跨环境精确恢复")
        c.raises(ValueError, lambda: restore_trainer(cut, dev, data=(tr[::-1], va)), "拒绝已改变数据")
        bad = cpu_copy(cut); bad["parameter_names"] = list(reversed(bad["parameter_names"]))
        c.raises(ValueError, lambda: restore_trainer(bad, dev), "拒绝参数顺序错配")
        bad = cpu_copy(cut); bad["scheduler"]["last_epoch"] = 999
        c.raises(ValueError, lambda: restore_trainer(bad, dev), "拒绝 scheduler/进度矛盾")
        bad = cpu_copy(cut); bad["scaler"] = {"scale": 128.0}
        c.raises(ValueError, lambda: restore_trainer(bad, dev), "拒绝 scaler/精度矛盾")
        bad = cpu_copy(cut); bad["boundary"] = "middle"
        c.raises(ValueError, lambda: restore_trainer(bad, dev), "拒绝中途边界标记")

        with tempfile.TemporaryDirectory() as tmp:
            path = Path(tmp)/"latest.pt"
            atomic_torch_save(cut, path)
            loaded = load_checkpoint(path)
            c.check(tree_equal(cut, loaded), "weights_only=True 完整加载基础容器")
            before_bytes = path.read_bytes()
            def fail_save(payload, file):
                file.write(b"partial")
                raise OSError("模拟写入失败")
            c.raises(OSError, lambda: atomic_torch_save(full, path, fail_save), "模拟保存中断报错")
            c.check(path.read_bytes() == before_bytes, "保存失败保留旧检查点")
            c.check(not list(Path(tmp).glob("*.tmp")), "失败临时文件清理")
            atomic_torch_save(full, path)
            c.check(tree_equal(load_checkpoint(path), full), "成功替换检查点")
            c.raises(FileNotFoundError, lambda: load_checkpoint(Path(tmp)/"absent.pt"), "缺失文件明确报错")

        scaler = scaler_resume_demo()
        c.check(scaler["all_exact"], "GradScaler 非空状态恢复")
        c.check(scaler["continuous_tail"] == scaler["resumed_tail"], "GradScaler 后续轨迹一致")
        c.check(not scaler["first_two"][1]["changed"], "非有限梯度跳过更新")
        c.check(scaler["restored_scale"] != scaler["fresh_scale"], "新 scaler 不等于已训练 scaler")
        for fault in ("optimizer", "rng", "cursor", "scheduler"):
            bad = restore_trainer(cut, dev, _fault=fault)
            bad.train_to(9)
            c.check(tensor_diff(full["model"], cpu_copy(bad.model.state_dict())) > 0,
                    f"漏恢复 {fault} 被参数对照发现")
    return {"passed": len(c.names), "tests": c.names, "device": "cpu",
            "cuda_tested": False, "note": "CUDA 仅在用户明确选择时另做完整实验"}

In [22]:
check_result = run_checks()
print("通过固定检查：", check_result["passed"], "项")
print("GPU 实测标记：", check_result["cuda_tested"])

通过固定检查： 72 项
GPU 实测标记： False


### 在终端用三个独立进程验证

下载配套 `.py` 后，在它所在目录执行下面三条命令。每一条结束之后再执行下一条。

```bash
# 第一个进程：累计训练到第7个窗口，正常保存后退出。
python lesson_26_training_checkpoints_and_resume.py --train --stop-after 7 --output-dir run_26_part

# 第二个进程：读取前一进程的完整状态，累计训练到第16个窗口。
python lesson_26_training_checkpoints_and_resume.py --resume run_26_part/latest.pt --total-windows 16 --output-dir run_26_resumed

# 第三个进程：同配置连续训练16个窗口，作为基准。
python lesson_26_training_checkpoints_and_resume.py --train --total-windows 16 --output-dir run_26_continuous
```

**`--total-windows 16` 表示累计总数为16，不是额外训练16次。**
输入检查点与新输出 `latest.pt` 不可同路径，本课主动保留输入文件。
没有使用“强行杀进程”的演示；若在窗口内部被打断，只能回到最近一次成功保存的边界，重放之后的工作。

可选 GPU：从第一条命令起使用同一套 `--device cuda --precision amp_fp16`，恢复时保持精度与环境一致。
本机遇到不支持的确定性操作会报错，不会悄悄降级后声称精确。

### `latest` 与 `best` 不同

`latest` 用于续跑，应保存最近的完整可恢复状态。`best` 通常按验证指标选出某个历史模型用于评估/部署。
如果维护 `best`、early stopping、EMA 或其他跨步状态，这些也需要恢复；本课没有启用它们。

不要把“best 权重”与“latest 优化器状态”拼在一起后称为断点恢复。
本课的输出包括：`interrupt.pt`、`continuous.pt`、`resumed.pt`、`experiment.json`、`history.json`。
手动训练模式生成 `latest.pt` 与 `training.json`。同名输出会覆盖。

## 11. 独立练习 A：保存并恢复随机状态

实现 `capture_rng_ex(device)` 与 `restore_rng_ex(state, device)`。
键与本课约定一致：`python`、`torch_cpu`、`torch_cuda`；CPU 张量要复制；CUDA 仅在 device 为 CUDA 时保存。
不能用“重新设 seed”代替恢复状态。

测试会先制造随机消耗，再核对下一组随机数，并验证没有修改保存的状态。

In [23]:
RUN_EXERCISE_A = False

def capture_rng_ex(device):
    raise NotImplementedError("请保存各个使用中的 RNG 当前状态")

def restore_rng_ex(state, device):
    raise NotImplementedError("请恢复状态，不要重新设置初始 seed")

In [24]:
def check_exercise_a(capture_fn, restore_fn):
    c = Checks()
    dev = torch.device("cpu")
    with controlled_environment(dev):
        seed_all(2606)
        torch.rand(7); random.random()
        state = capture_fn(dev)
        frozen = cpu_copy(state)
        expected = (random.random(), torch.rand(5))
        torch.rand(11); random.random()
        restore_fn(state, dev)
        actual = (random.random(), torch.rand(5))
        c.check(expected[0] == actual[0], "A Python 随机续接")
        c.check(torch.equal(expected[1], actual[1]), "A torch 随机续接")
        c.check(tree_equal(state, frozen), "A 不修改已保存状态")
        c.check(set(state) == {"python", "torch_cpu", "torch_cuda"}, "A 状态结构")
        c.check(state["torch_cuda"] == [], "A CPU 不伪造 CUDA 状态")
    return len(c.names)

if RUN_EXERCISE_A:
    print("A 通过：", check_exercise_a(capture_rng_ex, restore_rng_ex))

## 12. 独立练习 B：恢复数据游标

实现 `restore_cursor_ex(size, state)`，返回恢复后的 `DataCursor`。
可以调用 `DataCursor.load_state_dict()`，但不要只设置 epoch/cursor 或只重新设 seed。
测试会跨过当前 epoch 的末尾，继续检查下一轮的数据顺序，并检查非法排列会被拒绝。

In [25]:
RUN_EXERCISE_B = False

def restore_cursor_ex(size, state):
    raise NotImplementedError("请恢复当前 order/cursor/epoch 以及下一轮的 generator")


def check_exercise_b(fn):
    c = Checks()
    original = DataCursor(43, 2606)
    original.take_window(3, 4)
    original.take_window(3, 4)
    state = original.state_dict()
    restored = fn(43, state)
    c.check(tree_equal(original.state_dict(), restored.state_dict()), "B 完整状态恢复")
    for i in range(7):
        c.check(original.take_window(3, 4) == restored.take_window(3, 4), f"B 后续窗口 {i+1}")
    invalid = cpu_copy(state)
    invalid["order"][0] = invalid["order"][1]
    c.raises(ValueError, lambda: fn(43, invalid), "B 拒绝重复编号")
    c.check(state["cursor"] == 24, "B 不修改保存的游标")
    return len(c.names)

if RUN_EXERCISE_B:
    print("B 通过：", check_exercise_b(restore_cursor_ex))

## 13. 独立练习 C：组装完整恢复流程

实现 `resume_training_ex(payload, target_windows, device)`，返回恢复并训练到目标位置的 `Trainer`。
可以复用 `validate_payload`、`Trainer`、状态加载 API 和 A/B 的实现，但这道练习**不要直接调用 `restore_trainer`**。

重点检查：scheduler 已构造、模型参数顺序、优化器/scaler/数据/进度都恢复、RNG 最后恢复、模式和计数正确。
测试包含 Dropout、激活检查点与跨 epoch 的数据续接，并比较所有状态。

In [26]:
RUN_EXERCISE_C = False

def resume_training_ex(payload, target_windows, device):
    raise NotImplementedError("请按正确顺序恢复后继续训练")


def check_exercise_c(fn):
    c = Checks()
    dev = torch.device("cpu")
    with controlled_environment(dev):
        cfg = ModelConfig(d_model=16, n_heads=2, n_kv_heads=1, ff_dim=32, dropout=0.25)
        original = Trainer(cfg, TrainConfig(), dev)
        original.train_to(5)
        cut = original.snapshot()
        frozen = cpu_copy(cut)
        original.train_to(9)
        expected = original.snapshot()
        torch.rand(29); random.random()
        restored = fn(cut, 9, dev)
        actual = restored.snapshot()
        for key in ("model", "optimizer", "scheduler", "scaler", "rng", "data_cursor", "progress", "history"):
            c.check(tree_equal(actual[key], expected[key]), "C "+key)
        c.check(tree_equal(cut, frozen), "C 不修改传入快照")
        c.check(all(p.grad is None for p in restored.model.parameters()), "C 最终处于干净边界")
    return len(c.names)

if RUN_EXERCISE_C:
    print("C 通过：", check_exercise_c(resume_training_ex))

---
## 14. 参考答案：完成自己的实现后再看

下面使用新的函数名，不会覆盖你的练习。参考检查自动运行，是核对答案与测试的契约，不能替代你亲手实现。

In [27]:
def capture_rng_ref(device):
    return {
        "python": random.getstate(),
        "torch_cpu": torch.get_rng_state().clone(),
        "torch_cuda": [x.clone() for x in torch.cuda.get_rng_state_all()] if device.type == "cuda" else [],
    }

def restore_rng_ref(state, device):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch_cpu"].cpu())
    if device.type == "cuda":
        if len(state["torch_cuda"]) != torch.cuda.device_count():
            raise ValueError("CUDA 设备数不一致")
        torch.cuda.set_rng_state_all([x.cpu() for x in state["torch_cuda"]])

print("A 参考答案检查：", check_exercise_a(capture_rng_ref, restore_rng_ref))

A 参考答案检查： 5


In [28]:
def restore_cursor_ref(size, state):
    restored = DataCursor(size, seed=1)
    restored.load_state_dict(state)
    return restored

print("B 参考答案检查：", check_exercise_b(restore_cursor_ref))

B 参考答案检查： 10


In [29]:
def resume_training_ref(payload, target_windows, device):
    model_cfg, train_cfg = validate_payload(payload, device)
    trainer = Trainer(model_cfg, train_cfg, device)
    if [name for name, _ in trainer.model.named_parameters()] != payload["parameter_names"]:
        raise ValueError("参数注册顺序变化")
    trainer.model.load_state_dict(payload["model"], strict=True)
    trainer.model.train(payload["model_training"])
    trainer.scheduler.load_state_dict(copy.deepcopy(payload["scheduler"]))
    trainer.optimizer.load_state_dict(copy.deepcopy(payload["optimizer"]))
    trainer.scaler.load_state_dict(copy.deepcopy(payload["scaler"]))
    trainer.cursor.load_state_dict(payload["data_cursor"])
    trainer.progress = copy.deepcopy(payload["progress"])
    trainer.history = copy.deepcopy(payload["history"])
    trainer.optimizer.actual_steps = trainer.progress["updates_done"]
    trainer.optimizer.zero_grad(set_to_none=True)
    restore_rng_ref(payload["rng"], device)  # 对象已建好，最后恢复
    trainer.train_to(target_windows)
    return trainer

print("C 参考答案检查：", check_exercise_c(resume_training_ref))

C 参考答案检查： 10


## 15. 本课边界与工程迁移

**同一受控环境中的实测一致，不等于任何设备、版本、内核都逐位一致。** PyTorch 官方明确不保证跨版本和平台完全复现。[S4]
本课严格比较保存的环境字段；这些字段是排错线索，也不是所有底层软件状态的完整证明。

真实项目还可能需要保存：NumPy/第三方 RNG、分布式各 rank 状态、sampler/worker/stream 游标、动态分桶、学习率计划的总长度、EMA、最佳验证指标、早停计数、tokenizer 与预处理版本等。

本课为了可验证，主动收窄为：

- 同一份固定数据与顺序签名；无随机数据增强；
- 单进程显式游标，固定 micro-batch/累积策略；
- math 注意力后端、可记录的精度与确定性设置；
- 同步保存、单写入者、完整窗口边界；
- 不恢复尚未提交的梯度，也不支持动态迁移精度后仍称“精确恢复”。

**下一课：分词器与 BPE。** 从字符词表走向真实模型输入，学习词表、特殊 token、编码/解码与训练数据预处理边界。

## 原始资料

[S1] [PyTorch — Saving and Loading Models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)：一般检查点、优化器、参数快照。

[S2] [Optimizer.load_state_dict](https://docs.pytorch.org/docs/2.10/generated/torch.optim.Optimizer.load_state_dict.html)：优化器与 scheduler 加载顺序。

[S3] [AMP recipe — Saving/Resuming](https://docs.pytorch.org/tutorials/recipes/recipes/amp_recipe.html)：GradScaler 状态与保存时机。

[S4] [Reproducibility](https://docs.pytorch.org/docs/2.10/notes/randomness.html)：随机数、确定性与跨环境限制。

[S5] [torch.get_rng_state](https://docs.pytorch.org/docs/stable/generated/torch.get_rng_state.html)：CPU RNG 状态。

[S6] [torch.utils.data](https://docs.pytorch.org/docs/2.10/data.html)：数据加载与 worker 随机行为。

[S7] [AMP examples](https://docs.pytorch.org/docs/2.10/notes/amp_examples.html)：梯度累积的缩放、去缩放和更新时机。

[S8] [Python os.replace](https://docs.python.org/3/library/os.html#os.replace)：同文件系统替换语义。

[S9] [Serialization semantics](https://docs.pytorch.org/docs/stable/notes/serialization.html)：`weights_only` 的对象限制与安全边界。

脚本和讲义都以本次运行环境实际执行结果为准。源文档可能随版本更新；改变环境后应重新运行检查，不能复制旧报告作为新环境结果。